## STEP 1: 라이브러리

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score
)


sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 100

## STEP 2 : csv 파일 업로드

In [ ]:
import pandas as pd
df = pd.read_csv('../data/food_delivery_analytics_cleaned.csv')
df.head()

In [ ]:
print(f"Shape: {df.shape}")
print("\n--- Data Types & Nulls ---")
df.info()
df.describe().round(2)

## STEP 3 : 결측도 및 데이터 검사

In [ ]:
# Drop ID column
df.drop(columns=['order_id'], inplace=True)

# Fill numeric NAs with median
num_cols = df.select_dtypes(include='number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill boolean NAs with mode
bool_cols = df.select_dtypes(include='bool').columns
for col in bool_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print(f"Nulls remaining: {df.isnull().sum().sum()}")

## STEP 4 : 데이터 시각화

In [ ]:
counts = df['delayed_delivery_flag'].value_counts()
pct    = df['delayed_delivery_flag'].value_counts(normalize=True) * 100

summary = pd.DataFrame({'Count': counts, '%': pct.round(1)})
summary.index = ['On Time', 'Delayed']
print(summary)

counts.plot(kind='bar', color=['#2ecc71', '#e74c3c'], edgecolor='white')
plt.title('Delayed vs On-Time Deliveries')
plt.xticks([0, 1], ['On Time', 'Delayed'], rotation=0)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
features_to_plot = [
    'traffic_level_score',
    'delivery_distance_km',
    'preparation_time_minutes',
    'delivery_efficiency_score'
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, feat in zip(axes.flatten(), features_to_plot):
    sns.boxplot(
        data=df, x='delayed_delivery_flag', y=feat,
        palette=['#2ecc71', '#e74c3c'], ax=ax
    )
    ax.set_xticklabels(['On Time', 'Delayed'])
    ax.set_title(feat)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(24, 18))  # was (18, 12)
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, linewidths=0.5,
    annot_kws={'size': 9},  # adjust font size inside cells
    ax=ax
)
ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
ax.tick_params(axis='x', labelsize=10, rotation=45)
ax.tick_params(axis='y', labelsize=10)
plt.tight_layout()
plt.show()

## STEP 5 : Feature Engineering

In [ ]:
# Feature 1: stress score
# High traffic + bad weather = more stress on delivery
df['stress_score'] = df['traffic_level_score'] * df['weather_severity_score']

# Feature 2: Is it a peak hour?
df['is_peak_hour'] = df['order_hour'].between(11, 14) | df['order_hour'].between(18, 22)
df['is_peak_hour'] = df['is_peak_hour'].astype(int)

# Feature 3: Speed of partner (km per minute)
df['delivery_speed'] = df['delivery_distance_km'] / (df['delivery_time_minutes'] + 1)

In [ ]:
# ── Verify new features were added correctly ─────────────────
print("=== New Feature Sample ===")
print(df[['traffic_level_score', 'weather_severity_score', 'stress_score',
          'order_hour', 'is_peak_hour',
          'delivery_distance_km', 'delivery_time_minutes', 'delivery_speed']].head(5).round(3))

In [ ]:
# Boolean을 int 변환
for col in df.select_dtypes(include='bool').columns:
    df[col] = df[col].astype(int)

# Features (X) , Target (y) 정의
# TARGET = 'delayed_delivery_flag'
# X = df.drop(columns=[TARGET])
# y = df[TARGET]

drop_cols = ['delayed_delivery_flag', 'cancellation_flag', 'refund_flag']
X = df.drop(columns=drop_cols, errors='ignore')
y = df['delayed_delivery_flag']

print(f"Features (X) : {X.shape[1]} columns")
print(f"Target   (y) : {y.value_counts().to_dict()}")

# Train Data, Test Data 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Train set : {X_train.shape[0]:,} samples")
print(f"Test  set : {X_test.shape[0]:,} samples")
print(f"\ny_train distribution : {pd.Series(y_train).value_counts().to_dict()}")
print(f"y_test  distribution : {y_test.value_counts().to_dict()}")

# SMOTE
# smote = SMOTE(random_state=42)
# X_train, y_train = smote.fit_resample(X_train, y_train)

# print(f"y_train after SMOTE : {pd.Series(y_train).value_counts().to_dict()}")
# print("=" * 40)
# print("SUMMARY")
# print("=" * 40)
# print(f"X_train : {X_train.shape}")
# print(f"X_test  : {X_test.shape}")
# print(f"y_train : {pd.Series(y_train).value_counts().to_dict()}  ← balanced (SMOTE)")
# print(f"y_test  : {y_test.value_counts().to_dict()}  ← real distribution")
# print(f"\nTotal Features : {X_train.shape[1]}")
# print(f"\nFeature List:")
# for col in X.columns:
#     print(f"  → {col}")

#STEP 6 : 모델 구현 및 훈련

In [ ]:
# KNN 모델을 위한 스케일링 작업
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

In [ ]:
# KNN 모델 학습
print("\n[KNN 모델 학습 시작]")
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_sc, y_train)
print("\n[KNN 모델 학습 완료!]")

In [ ]:
# XGBoost 모델 학습
weight = y_train.value_counts()[0] / y_train.value_counts()[1]

print("\n[XGBoost 모델 학습 시작]")
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    scale_pos_weight=weight, # 정상 배달보다 적은 수의 지연 배달을 맞추면 점수를 더 주어 데이터의 불균형을 해소
    eval_metric='logloss'
)
xgb.fit(X_train, y_train)
print("\n[XGBoost 모델 학습 완료!]")

#STEP 7 : 모델 성능 평가

In [ ]:
# 모델 평가 함수 정의
def evaluate_model (name, model, X_test_data, y_test_data):
  y_pred = model.predict(X_test_data)
  y_pred_prob = model.predict_proba(X_test_data)[:, 1]

  acc = accuracy_score(y_test_data, y_pred)
  auc = roc_auc_score(y_test_data, y_pred_prob)

  print(f'\n{'='*50}')
  print(f'📌 {name}')
  print(f'{'='*50}')
  print(f'  Accuracy : {acc:.4f}  ({acc*100:.2f}%)')
  print(f'  AUC-ROC  : {auc:.4f}')
  print()
  print(classification_report(y_test_data, y_pred, target_names=['On Time', 'Delayed']))

  return auc, y_pred_prob

In [ ]:
auc_knn, prob_knn = evaluate_model('KNN', knn, X_test_sc, y_test)
auc_xgb, prob_xgb = evaluate_model('XGBoost', xgb, X_test, y_test)

In [ ]:
# ROC 곡선 비교, AUC 점수 비교
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: ROC Curves ---
for name, prob, auc in [
    ('KNN', prob_knn, auc_knn),
    ('XGBoost', prob_xgb, auc_xgb)
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', label='Random Guess')
axes[0].set_title('ROC Curve — All Models', fontsize=13, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

# --- Plot 2: AUC Score Bar Chart ---
models = ['KNN', 'XGBoost']
scores = [auc_knn, auc_xgb]
colors = ['#3498db', '#2ecc71']
bars   = axes[1].bar(models, scores, color=colors, edgecolor='white', width=0.5)

axes[1].set_ylim(0, 1.0)
axes[1].set_title('AUC-ROC Score Comparison', fontsize=13, fontweight='bold')
axes[1].set_ylabel('AUC-ROC Score')

for bar, score in zip(bars, scores):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{score:.3f}', ha='center', fontweight='bold'
    )

plt.tight_layout()
plt.show()

In [ ]:
# 더 정확한 XGBoost 모델에 대한 혼동 행렬 적용
best_pred = xgb.predict(X_test)
cm = confusion_matrix(y_test, best_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['On Time', 'Delayed'],
    yticklabels=['On Time', 'Delayed'], ax=ax
)
ax.set_title('Confusion Matrix — XGBoost Model (Best Model)', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('Actual Label')
plt.tight_layout()
plt.show()

In [ ]:
importance_df = pd.DataFrame({
    'Feature'  : X_train.columns,
    'Importance': xgb.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=importance_df, x='Importance', y='Feature',
    palette='viridis', ax=ax
)
ax.set_title('Top 15 Features That Predict Delivery Delays', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()